# DocLayNet OCR Benchmark + 10-Epoch Finetune

Workflow: clone repo from GitHub, benchmark all runnable OCR engines, prepare/run finetune, then benchmark the finetuned PaddleOCR model when available.

In [ ]:
import os, shutil, subprocess, sys

GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/ocr_benchmark.git'
REPO_DIR = '/kaggle/working/ocr_benchmark'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('repo:', REPO_DIR)

In [ ]:
!python -m pip install -q -r requirements.txt

## Install all benchmark engines

PaddleOCR-VL and Surya are heavy. If Kaggle runs out of GPU memory, restart the session and run fewer engines at a time with the same config/seed.

In [ ]:
!python -m pip install -q -r requirements-kaggle-docling.txt
!python -m pip install -q -r requirements-kaggle-paddleocr.txt
!python -m pip install -q -r requirements-kaggle-surya.txt

# PaddleOCR-VL is the heaviest install. Keep this enabled for the full all-model run.
!python -m pip install -q -r requirements-kaggle-paddleocr-vl.txt

In [ ]:
CONFIG = 'configs/kaggle_doclaynet_science.yaml'
BENCH_OUT = '/kaggle/working/ocr_benchmark_outputs'
FT_OUT = '/kaggle/working/ocr_finetune_outputs'
LIMIT = 20
EPOCHS = 10
ALL_PRETRAINED_ENGINES = 'docling paddleocr_vl paddleocr surya omnidocbench protonx_legal_tc'

In [ ]:
# Smoke test: verifies dataset loading, metrics, reporting, and skipped non-OCR entries.
!PYTHONPATH=src python -m ocr_benchmark.benchmark \
  --config {CONFIG} \
  --engines noop omnidocbench protonx_legal_tc \
  --limit 3 \
  --output-dir {BENCH_OUT}/smoke

In [ ]:
# Benchmark all pretrained runnable models, plus skipped reference/non-OCR links.
!PYTHONPATH=src python -m ocr_benchmark.benchmark \
  --config {CONFIG} \
  --engines {ALL_PRETRAINED_ENGINES} \
  --limit {LIMIT} \
  --output-dir {BENCH_OUT}/pretrained_all

## Finetune 10 epochs

This repo runs Kaggle-feasible finetuning for PaddleOCR recognition. Docling, Surya, PaddleOCR-VL, OmniDocBench, and protonx are recorded in `finetune_status.csv` with the reason they are not trained by this single-notebook path.

In [ ]:
# Prepare finetune data and commands for every listed model.
!PYTHONPATH=src python -m ocr_benchmark.finetune \
  --config {CONFIG} \
  --models paddleocr docling paddleocr_vl surya omnidocbench protonx_legal_tc \
  --limit {LIMIT} \
  --epochs {EPOCHS} \
  --output-dir {FT_OUT}

In [ ]:
# Run actual PaddleOCR finetuning + export. This can take a while.
# If it fails due to Paddle package/CUDA constraints, the pretrained benchmark above is still complete.
!PYTHONPATH=src python -m ocr_benchmark.finetune \
  --config {CONFIG} \
  --models paddleocr docling paddleocr_vl surya omnidocbench protonx_legal_tc \
  --limit {LIMIT} \
  --epochs {EPOCHS} \
  --execute \
  --output-dir {FT_OUT}

In [ ]:
# Benchmark finetuned PaddleOCR if export exists.
import os
ft_model_dir = f'{FT_OUT}/paddleocr/inference'
if os.path.exists(ft_model_dir):
    !PYTHONPATH=src python -m ocr_benchmark.benchmark \
      --config {CONFIG} \
      --engines paddleocr_ft \
      --limit {LIMIT} \
      --output-dir {BENCH_OUT}/finetuned_paddleocr
else:
    print('No exported finetuned PaddleOCR inference model found:', ft_model_dir)

In [ ]:
import pandas as pd
from pathlib import Path

tables = []
for name, path in [
    ('pretrained_all', f'{BENCH_OUT}/pretrained_all/summary.csv'),
    ('finetuned_paddleocr', f'{BENCH_OUT}/finetuned_paddleocr/summary.csv'),
]:
    if Path(path).exists():
        df = pd.read_csv(path)
        df.insert(0, 'run', name)
        tables.append(df)

combined = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
display(combined)
combined.to_csv('/kaggle/working/combined_benchmark_summary.csv', index=False)

ft_status_path = f'{FT_OUT}/finetune_status.csv'
if Path(ft_status_path).exists():
    display(pd.read_csv(ft_status_path))

In [ ]:
!cd /kaggle/working && zip -qr ocr_doclaynet_results.zip ocr_benchmark_outputs ocr_finetune_outputs combined_benchmark_summary.csv
print('/kaggle/working/ocr_doclaynet_results.zip')